**姓名：**

**学号：**

**班级：**

# 课后作业 · 用朴素贝叶斯筛选校园邮件

《数据分析与机器学习》｜建议用时 35–45 分钟

你要给校园邮箱做一个过滤器，根据邮件里的用词把每封邮件判为「正常」或「垃圾」。训练模型后，用 20 封测试邮件看看它拦错了哪些邮件。

在 Jupyter 或 Colab 中按顺序运行。数据和关键词编码已准备好。从任务一开始补全标有 **TODO** 的代码，在「我的答案」处作答，保留运行输出并提交这份 `.ipynb` 文件。

## 1. 认识邮件和标签

下面是 80 封仿真的校园邮件。每条记录有两项，分别是邮件文本和类别标签。

- `0` 代表正常邮件，包括课程通知、日常往来和收件人已订阅的推广。
- `1` 代表垃圾邮件，这里指收件人未订阅、却被群发的推广或活动信息。

标签根据仿真的收件情况确定，例如收件人是否订阅了这类邮件。正常邮件也可能提到「免费」，垃圾邮件也可能写「请查收」。

用 60 封训练邮件学习用词规律，再用另外 20 封测试邮件检查效果。

In [ ]:
train_mails = [
    ('请查收材料力学课程的答疑会议纪要。', 0),
    ('明天的数据分析课改到教学楼 B204，请提前五分钟到场。', 0),
    ('线性代数第二次作业已经批改，请查收批注。', 0),
    ('实验室本周五进行例行检修，下午暂停开放。', 0),
    ('校图书馆周六有一场免费讲座，主题是地方志的整理。', 0),
    ('课程小组会议改到周三晚八点，教室照旧。', 0),
    ('请查收 11 月图书借阅清单，到期前可续借。', 0),
    ('今天的课件已放到课程平台第九周目录。', 0),
    ('学校心理中心为本校学生提供免费咨询，预约时请选择校区。', 0),
    ('学院教学工作会议周四举行，各班请派一位代表。', 0),
    ('校园卡服务台搬到学生事务大厅一楼。', 0),
    ('请查收下周实验分组，周一上课前核对学号。', 0),
    ('毕业设计开题登记表可在教务系统下载。', 0),
    ('校运会志愿者限时补录，报名表今晚九点关闭。', 0),
    ('我整理了小组访谈提纲，请查收修改稿。', 0),
    ('周日还去操场练习吗？我上午十点以后有空。', 0),
    ('昨天落在机房的水杯已交到前台。', 0),
    ('请查收你上次借我的书单，第二本图书馆还有库存。', 0),
    ('社团招新会议定在活动室二楼，策划组先到。', 0),
    ('我打印好了展示海报，午饭后带到教室。', 0),
    ('谢谢你帮忙核对测量记录，第三组数据已经更新。', 0),
    ('会议前能帮我看一眼发言提纲吗？', 0),
    ('图书馆三楼靠窗的位置有空，如果来复习我在这边。', 0),
    ('请查收你让我帮忙补记的两页课堂笔记。', 0),
    ('器材借用期限延到周六，我会负责归还。', 0),
    ('请查收本周志愿服务名单，集合时间提前了十分钟。', 0),
    ('昨天读书会的合照我放在共享相册里了。', 0),
    ('线上会议我可能晚到一会儿，请先讨论分工。', 0),
    ('你订阅的校出版社新书简报已更新，退订方式见邮件末尾。', 0),
    ('校友书店给订阅读者准备了免费试读章节。', 0),
    ('本期社区剧场订阅简报已寄出，如需退订可修改接收偏好。', 0),
    ('校友活动订户可以免费预约本月展览，退订请更新偏好。', 0),
    ('校园影展会员简报公布了周末场次，退订入口在账户设置。', 0),
    ('你订阅的语言学习简报本周介绍三本法语入门书。', 0),
    ('校园文创店订户可参加本周限时团购，退订方式见会员页。', 0),
    ('恭喜你在上周报名的读书会抽奖中中奖，可在活动现场领取纪念品。', 0),
    ('你订阅的科普月刊本期开放免费阅读。', 0),
    ('请查收你订阅的校园讲座月历，退订请调整通知选项。', 0),
    ('你关注的校友市集公布了摊位名单和开放时段。', 0),
    ('订阅书店简报的读者可以免费领取一份本月书目。', 0),
    ('校内健身房订阅简报公布了会员限时续费安排。', 0),
    ('你订阅的学生剧团简报附有排练花絮，退订可在个人中心操作。', 0),
    ('你已中奖，本周限时领取惊喜礼包。', 1),
    ('迎新大促限时开放，新用户可免费领取礼盒。', 1),
    ('会员专场免费开放，今日限时领取购物券。', 1),
    ('你的号码已中奖，免费奖品今日开放领取。', 1),
    ('恭喜中奖，请查收本期活动领奖通知。', 1),
    ('限时免费试听速成课程，体验名额今日发放。', 1),
    ('请查收中奖回执，免费礼包将在活动期间开放领取。', 1),
    ('限时领取购物券，中奖名单已公布。', 1),
    ('你已入选本期中奖名单，奖品由活动方统一发放。', 1),
    ('年终特惠限时开放，会员礼包免费派送。', 1),
    ('请查收本期折扣券，会员专享活动已经开启。', 1),
    ('恭喜中奖，免费兑换券本月有效。', 1),
    ('快来参加本周限时特惠，订单附赠纪念礼品。', 1),
    ('中奖会员专享免费礼包，活动到今晚结束。', 1),
    ('今日限时推广，免费送试用装，退订说明见会员栏。', 1),
    ('系统随机抽取了获奖收件人，你的礼包正在等待确认。', 1),
    ('本期教育优惠限时公布，欢迎查收会员报名通知。', 1),
    ('本周免费领取试听券，欢迎参加体验营。', 1),
]

In [ ]:
test_mails = [
    ('数据分析课程的答疑会议改在周五下午，请查收更新后的安排。', 0),
    ('校图书馆的论文写作讲座限时开放报名，本校师生免费参加。', 0),
    ('恭喜中奖，新会员礼盒可限时免费领取。', 1),
    ('你订阅的校园影展简报来了，退订请修改账户设置。', 0),
    ('我把实验报告的图表重画了一遍，明天在教室给你。', 0),
    ('请查收本店本周的会员优惠清单。', 1),
    ('请查收教务处发出的补考教室变更通知。', 0),
    ('今晚限时专场，免费发放购物抵用券。', 1),
    ('周五课程取消，补课日期会在下周公布。', 0),
    ('你订阅的校友讲堂简报公布了下周会议安排，退订方式见账户设置。', 0),
    ('请查收你订阅的校出版社目录，本月开放免费试读。', 0),
    ('本期活动中奖结果已公布，请查收领取通知。', 1),
    ('我们周三在校门口集合，我会带上借来的相机。', 0),
    ('校内篮球联赛限时补报，明天中午系统关闭。', 0),
    ('本店新品预售已经开始，会员价本周有效。', 1),
    ('学院实习说明会议移到北楼 302，具体时间照旧。', 0),
    ('老师已在课程平台更新阅读材料和参考书目。', 0),
    ('免费试用装限时发放，中奖会员可领大礼盒。', 1),
    ('请查收新版实验安全须知，下次上机前阅读。', 0),
    ('上次你借我的计算器我已交给值班同学，周一可以取。', 0),
]

In [ ]:
print("训练邮件", len(train_mails), "封")
print("测试邮件", len(test_mails), "封")

for text, label in train_mails[:3]:
    print(f"标签 {label}　{text}")

## 2. 准备关键词特征

沿用课堂的规则。一个关键词在邮件中出现就记 `1`，没有出现就记 `0`。同一个词出现几次都只记一个 `1`。

这次有六个关键词。每封邮件都按相同的顺序编码。

| 关键词 | 免费 | 中奖 | 限时 | 查收 | 会议 | 退订 |
| --- | --- | --- | --- | --- | --- | --- |
| 「请查收材料力学课程的答疑会议纪要。」 | 0 | 0 | 0 | 1 | 1 | 0 |

直接运行下面的代码，为训练和预测准备特征。

In [ ]:
keywords = ["免费", "中奖", "限时", "查收", "会议", "退订"]


def encode_mail(text):
    features = []
    for word in keywords:
        if word in text:
            features.append(1)
        else:
            features.append(0)
    return features

下面的代码生成训练和测试数据。`X_train` 和 `X_test` 存放邮件的关键词编码，`y_train` 和 `y_test` 存放对应的类别标签。

In [ ]:
X_train = []
y_train = []
for text, label in train_mails:
    X_train.append(encode_mail(text))
    y_train.append(label)

X_test = []
y_test = []
for text, label in test_mails:
    X_test.append(encode_mail(text))
    y_test.append(label)

print("第一封训练邮件的特征", X_train[0])
print("第一封训练邮件的标签", y_train[0])

## 3. 任务一 · 训练模型并预测

这里的特征只有 `0` 和 `1`，所以用 `BernoulliNB`。设置 `alpha=1`，沿用商品推荐练习里的加一平滑。

补全 TODO 1 和 TODO 2。调用 `fit` 时，把训练特征和训练标签交给模型。调用 `predict` 时，只传入测试特征，得到 20 个预测标签。

In [ ]:
from sklearn.naive_bayes import BernoulliNB

model = BernoulliNB(alpha=1)
model.fit(..., ...)  # TODO 1：填入训练特征和训练标签
y_pred = model.predict(...)  # TODO 2：填入测试特征

print("预测结果", y_pred)

运行下面的代码，查看第 1 封测试邮件的预测。把预测类别和模型给垃圾类的概率记在下方。

In [ ]:
sample_index = 0
label_names = {0: "正常", 1: "垃圾"}

print("邮件", test_mails[sample_index][0])
print("关键词编码", X_test[sample_index])
print("预测类别", label_names[y_pred[sample_index]])

probabilities = model.predict_proba([X_test[sample_index]])[0]
for label, probability in zip(model.classes_, probabilities):
    print(f"{label_names[label]}邮件的模型概率 {probability:.1%}")

**我的答案**　第 1 封邮件被判为 ______，模型给垃圾类的概率是 ______。

## 4. 任务二 · 评价筛选结果

以垃圾邮件为正类。混淆矩阵的行是真实类别，列是预测类别。行和列都按「正常、垃圾」排序。

| 实际类别＼预测类别 | 正常 | 垃圾 |
| --- | --- | --- |
| 正常 | 正确放行（`tn`） | 误拦（`fp`） |
| 垃圾 | 漏拦（`fn`） | 正确拦截（`tp`） |

In [ ]:
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp = matrix[0]
fn, tp = matrix[1]

print(matrix)
print(f"正确放行 {tn} 封，误拦 {fp} 封，漏拦 {fn} 封，正确拦截 {tp} 封。")

### 4.1 计算指标

用上面的四个计数补全 TODO 3–5。先找出每个指标要统计哪一组邮件。

- **准确率**看所有测试邮件中判对的比例。
- **精确率**看被模型拦下的邮件中有多少确实是垃圾邮件。
- **召回率**看所有实际的垃圾邮件中有多少被模型拦下。

再把模型和「全部放行」比较。全部放行相当于把每封邮件都标成正常。下方已给出这种做法的准确率和召回率。

In [ ]:
accuracy = ...   # TODO 3：准确率
precision = ...  # TODO 4：垃圾类精确率
recall = ...     # TODO 5：垃圾类召回率

print(f"模型准确率 {accuracy:.1%}")
print(f"模型精确率 {precision:.1%}")
print(f"模型召回率 {recall:.1%}")

baseline_accuracy = y_test.count(0) / len(y_test)
baseline_recall = 0 / y_test.count(1)

print(f"全部放行的准确率 {baseline_accuracy:.1%}")
print(f"全部放行的召回率 {baseline_recall:.1%}")

**我的答案**　与全部放行相比，这个模型的准确率和垃圾类召回率各变化了多少个百分点？用本次结果解释，只看全部放行的准确率会忽略什么？

答：

### 4.2 分析误拦和漏拦

运行下面的代码，从输出里各选一封误拦和漏拦的邮件。对照关键词顺序，指出模型看到了哪些词。再想一想，如果你是收件人，还会看哪条信息来判断这封邮件？

In [ ]:
print("关键词顺序", keywords)
for index, (text, actual_label) in enumerate(test_mails):
    if actual_label == y_pred[index]:
        continue

    mistake = "误拦" if actual_label == 0 else "漏拦"
    print(f"\n第 {index + 1} 封（{mistake}）")
    print(text)
    print("关键词编码", X_test[index])

**我的答案**

- **误拦**　第 ______ 封。模型看到的关键词是 ______。我还会看 ______，因为 ______。
- **漏拦**　第 ______ 封。模型看到的关键词是 ______。我还会看 ______，因为 ______。

## 5. 任务三

如果校园邮箱要试用这个过滤器，你会怎么处理被判为垃圾的邮件？ 可以直接移到垃圾箱，也可以标记为「疑似垃圾」并留在收件箱，还可以提出自己的办法。请结合本次测试中的一封误拦邮件和一封漏拦邮件，用三五句话说明你的做法，以及它会给收件人带来什么好处和麻烦。

我的回答：